# Ensemble: weights by class


In [ ]:
!pip install ultralytics ensemble-boxes -q
from ultralytics import YOLO

In [ ]:
# emsemble ver.2 
# gave me too many warnings: zero area box 

# ── Install dependencies (new session) ──
!pip install ultralytics ensemble-boxes -q

from ultralytics import RTDETR, YOLO
import pandas as pd
import os
from ensemble_boxes import weighted_boxes_fusion

CLASS_NAMES = ['Aortic enlargement','Atelectasis','Calcification','Cardiomegaly',
               'Consolidation','ILD','Infiltration','Lung Opacity','Nodule/Mass',
               'Other lesion','Pleural effusion','Pleural thickening',
               'Pneumothorax','Pulmonary fibrosis']

# Per-class val mAP50 for each model (index-aligned with CLASS_NAMES)
# Source: RT-DETR values from project_status.md validation table;
# YOLOv8m values from this notebook's training/validation log output.
RTDETR_MAP50 = [0.481, 0.094, 0.151, 0.442, 0.243, 0.146, 0.195, 0.227,
                0.163, 0.112, 0.298, 0.194, 0.085, 0.243]
YOLO_MAP50   = [0.633, 0.077, 0.077, 0.607, 0.265, 0.205, 0.227, 0.210,
                0.184, 0.057, 0.298, 0.153, 0.016, 0.250]

# Compute per-class weights proportional to each model's val mAP50.
# Add a small epsilon to avoid divide-by-zero when both models score 0.
EPS = 1e-6
CLASS_WEIGHTS = {}
for i in range(14):
    rt, yo = RTDETR_MAP50[i], YOLO_MAP50[i]
    total = rt + yo + EPS
    CLASS_WEIGHTS[i] = [rt / total, yo / total]

def get_preds_by_class(model, image_id, img_dir, imgsz=512, conf=0.15):
    img_path = os.path.join(img_dir, f'{image_id}.png')
    results = model.predict(img_path, imgsz=imgsz, conf=conf, device=0, verbose=False)
    out = {c: {'boxes': [], 'scores': []} for c in range(14)}
    r = results[0]
    if r.boxes is not None and len(r.boxes) > 0:
        for box in r.boxes:
            cls = int(box.cls.item())
            if cls not in out:
                continue
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            # normalize based on imgsz=512
            # out[cls]['boxes'].append([x1/512, y1/512, x2/512, y2/512])
            out[cls]['boxes'].append([
            min(max(x1/512, 0), 1),
            min(max(y1/512, 0), 1),
            min(max(x2/512, 0), 1),
            min(max(y2/512, 0), 1)
            ])
            out[cls]['scores'].append(float(box.conf.item()))
    return out

def ensemble_image(rtdetr_model, yolo_model, image_id, img_dir, img_size=1024):
    rt_preds   = get_preds_by_class(rtdetr_model, image_id, img_dir)
    yolo_preds = get_preds_by_class(yolo_model,   image_id, img_dir)

    pred_strings = []
    for class_id in range(14):
        b1, s1 = rt_preds[class_id]['boxes'],   rt_preds[class_id]['scores']
        b2, s2 = yolo_preds[class_id]['boxes'], yolo_preds[class_id]['scores']

        boxes_list, scores_list, labels_list, weights = [], [], [], []
        w1, w2 = CLASS_WEIGHTS[class_id]

        if b1:
            boxes_list.append(b1)
            scores_list.append(s1)
            labels_list.append([class_id] * len(b1))
            weights.append(w1)
        if b2:
            boxes_list.append(b2)
            scores_list.append(s2)
            labels_list.append([class_id] * len(b2))
            weights.append(w2)

        if not boxes_list:
            continue

        boxes, scores, labels = weighted_boxes_fusion(
            boxes_list, scores_list, labels_list,
            weights=weights,
            iou_thr=0.5,
            skip_box_thr=0.0  # no double filtering
        )
        for box, score, label in zip(boxes, scores, labels):
            # scale back to 1024
            x1, y1, x2, y2 = [int(c * 1024) for c in box]
            pred_strings.append(f"{int(label)} {score:.4f} {x1} {y1} {x2} {y2}")

    return pred_strings

# ── Load both trained models ──
rtdetr_model = RTDETR('/kaggle/input/datasets/leehyunji0116/rtdetr-best-pt/rtdetr_best.pt')
yolo_model   = YOLO('/kaggle/input/datasets/leehyunji0116/yolo-best-pt/yolo_best.pt')

# ── Run ensemble inference over the full test set ──
DATA_DIR      = '/kaggle/input/competitions/amia-public-challenge-2026'
TEST_IMG_DIR  = f'{DATA_DIR}/test/test'

test_df  = pd.read_csv(f'{DATA_DIR}/test.csv')
test_ids = test_df['image_id'].unique()

submissions = []
for image_id in test_ids:
    pred_strings = ensemble_image(rtdetr_model, yolo_model, image_id, TEST_IMG_DIR)
    if not pred_strings:
        pred_strings = ["14 1.0 0 0 1 1"]  # "No finding" fallback
    submissions.append({'image_id': image_id, 'PredictionString': ' '.join(pred_strings)})

sub_df = pd.DataFrame(submissions)
sub_df.to_csv('/kaggle/working/classwise_ensemble_submission.csv', index=False)
print("Done!")

In [ ]:
# ensemble ver.3
# included clipping, (< 0.01) skip

!pip install ultralytics ensemble-boxes -q

from ultralytics import RTDETR, YOLO
import pandas as pd
import os
from ensemble_boxes import weighted_boxes_fusion
import warnings
warnings.filterwarnings('ignore')

CLASS_NAMES = ['Aortic enlargement','Atelectasis','Calcification','Cardiomegaly',
               'Consolidation','ILD','Infiltration','Lung Opacity','Nodule/Mass',
               'Other lesion','Pleural effusion','Pleural thickening',
               'Pneumothorax','Pulmonary fibrosis']

RTDETR_MAP50 = [0.481, 0.094, 0.151, 0.442, 0.243, 0.146, 0.195, 0.227,
                0.163, 0.112, 0.298, 0.194, 0.085, 0.243]
YOLO_MAP50   = [0.633, 0.077, 0.077, 0.607, 0.265, 0.205, 0.227, 0.210,
                0.184, 0.057, 0.298, 0.153, 0.016, 0.250]

EPS = 1e-6
CLASS_WEIGHTS = {}
for i in range(14):
    rt, yo = RTDETR_MAP50[i], YOLO_MAP50[i]
    total = rt + yo + EPS
    CLASS_WEIGHTS[i] = [rt / total, yo / total]

def get_preds_by_class(model, image_id, img_dir, imgsz=512, conf=0.15):
    img_path = os.path.join(img_dir, f'{image_id}.png')
    results = model.predict(img_path, imgsz=imgsz, conf=conf, device=0, verbose=False)
    out = {c: {'boxes': [], 'scores': []} for c in range(14)}
    r = results[0]
    if r.boxes is not None and len(r.boxes) > 0:
        for box in r.boxes:
            cls = int(box.cls.item())
            if cls not in out:
                continue
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            # normalize to 0~1
            x1 = min(max(x1/512, 0), 1)
            y1 = min(max(y1/512, 0), 1)
            x2 = min(max(x2/512, 0), 1)
            y2 = min(max(y2/512, 0), 1)
            # skip boxes that are too small after clipping (< 10px in 1024 space)
            if (x2 - x1) < 0.01 or (y2 - y1) < 0.01:
                continue
            out[cls]['boxes'].append([x1, y1, x2, y2])
            out[cls]['scores'].append(float(box.conf.item()))
    return out

def ensemble_image(rtdetr_model, yolo_model, image_id, img_dir, img_size=1024):
    rt_preds   = get_preds_by_class(rtdetr_model, image_id, img_dir)
    yolo_preds = get_preds_by_class(yolo_model,   image_id, img_dir)

    pred_strings = []
    for class_id in range(14):
        b1, s1 = rt_preds[class_id]['boxes'],   rt_preds[class_id]['scores']
        b2, s2 = yolo_preds[class_id]['boxes'], yolo_preds[class_id]['scores']

        boxes_list, scores_list, labels_list, weights = [], [], [], []
        w1, w2 = CLASS_WEIGHTS[class_id]

        if b1:
            boxes_list.append(b1)
            scores_list.append(s1)
            labels_list.append([class_id] * len(b1))
            weights.append(w1)
        if b2:
            boxes_list.append(b2)
            scores_list.append(s2)
            labels_list.append([class_id] * len(b2))
            weights.append(w2)

        if not boxes_list:
            continue

        boxes, scores, labels = weighted_boxes_fusion(
            boxes_list, scores_list, labels_list,
            weights=weights,
            iou_thr=0.5,
            skip_box_thr=0.0
        )
        for box, score, label in zip(boxes, scores, labels):
            x1, y1, x2, y2 = [int(c * 1024) for c in box]
            pred_strings.append(f"{int(label)} {score:.4f} {x1} {y1} {x2} {y2}")

    return pred_strings

# load models
rtdetr_model = RTDETR('/kaggle/input/datasets/leehyunji0116/rtdetr-best-pt/rtdetr_best.pt')
yolo_model   = YOLO('/kaggle/input/datasets/leehyunji0116/yolo-best-pt/yolo_best.pt')

DATA_DIR     = '/kaggle/input/competitions/amia-public-challenge-2026'
TEST_IMG_DIR = f'{DATA_DIR}/test/test'

test_df  = pd.read_csv(f'{DATA_DIR}/test.csv')
test_ids = test_df['image_id'].unique()

submissions = []
for i, image_id in enumerate(test_ids):
    pred_strings = ensemble_image(rtdetr_model, yolo_model, image_id, TEST_IMG_DIR)
    if not pred_strings:
        pred_strings = ["14 1.0 0 0 1 1"]
    submissions.append({'image_id': image_id, 'PredictionString': ' '.join(pred_strings)})
    if (i+1) % 500 == 0:
        print(f"Progress: {i+1}/{len(test_ids)}")

sub_df = pd.DataFrame(submissions)
sub_df.to_csv('/kaggle/working/classwise_ensemble_v3_submission.csv', index=False)
print("Done!")
print(sub_df.head())

!kaggle competitions submit -c amia-public-challenge-2026 -f /kaggle/working/classwise_ensemble_v3_submission.csv -m "Classwise ensemble v3 - fixed clipping and degenerate boxes"
print("Submitted!")

Progress: 500/6427
Progress: 1000/6427
Progress: 1500/6427
Progress: 2000/6427
Progress: 2500/6427
Progress: 3000/6427
Progress: 3500/6427
Progress: 4000/6427
Progress: 4500/6427
Progress: 5000/6427
Progress: 5500/6427
Progress: 6000/6427
Done!
                           image_id  \
0  3r9OdPSdvQ58qI3VUFUeSKyCvxBpFc0c   
1  LO2jAm8E96Ih87wJVoqiOXHixrwPMeOm   
2  PN7S4HbhNp4fht9TTc6DXGOKGkeRTR7W   
3  l7f2KDvrnrh26v4aYgi0Slj7lVBZMQIL   
4  if5Pqu95xLUtURzAo72YiSg8GNzJb1F3   

                                    PredictionString  
0  0 0.2279 982 577 1024 754 12 0.3102 154 364 84...  
1  7 0.1953 507 402 655 500 8 0.1583 515 403 654 ...  
2                                     14 1.0 0 0 1 1  
3                                     14 1.0 0 0 1 1  
4                                     14 1.0 0 0 1 1  
100%|████████████████████████████████████████| 454k/454k [00:00<00:00, 1.02MB/s]
Successfully submitted to AMIA Public Challenge 2026Submitted!


In [ ]:
!kaggle competitions submit -c amia-public-challenge-2026 -f /kaggle/working/classwise_ensemble_submission.csv -m "RT-DETR + YOLOv8m class-wise mAP-weighted WBF"

100%|████████████████████████████████████████| 656k/656k [00:00<00:00, 1.58MB/s]
Successfully submitted to AMIA Public Challenge 2026